In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"bf16 native: {torch.cuda.is_bf16_supported(including_emulation=False)}")

NVIDIA L4
VRAM: 23.7 GB
bf16 native: True


In [ ]:
#  !pip install -q pytest-asyncio

In [ ]:
# Repo Setup:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/akshatvishu/vllm-omni.git"
BRANCH = "feat/ming-omni-tts-dense"
REPO_DIR = "/content/vllm-omni"


def run(cmd, cwd=None, check=True):
    print(f"+ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if check and result.returncode != 0:
        raise RuntimeError(f"Failed ({result.returncode}): {cmd}")
    return result


def ensure_repo():
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        if os.path.exists(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
        return

    print("Repo already exists, refreshing checkout.")
    run(f"git remote set-url origin {REPO_URL}", cwd=REPO_DIR)
    run(f"git fetch --prune --depth 1 origin {BRANCH}", cwd=REPO_DIR)
    run(f"git checkout -B {BRANCH} FETCH_HEAD", cwd=REPO_DIR)
    run(f"git reset --hard FETCH_HEAD", cwd=REPO_DIR)
    run(f"git clean -fdx", cwd=REPO_DIR)




In [ ]:
ensure_repo()

print("\nACTIVE COMMIT:")
run("git log -1 --oneline", cwd=REPO_DIR)

+ git clone --depth 1 --branch feat/ming-omni-tts-dense https://github.com/akshatvishu/vllm-omni.git /content/vllm-omni
Cloning into '/content/vllm-omni'...

ACTIVE COMMIT:
+ git log -1 --oneline
050a927 Fix Ming stage0 sampled control cache for non-eager


CompletedProcess(args='git log -1 --oneline', returncode=0, stdout='050a927 Fix Ming stage0 sampled control cache for non-eager\n', stderr='')

In [ ]:
# Install deps
run(f'"{sys.executable}" -m pip install -U uv -q')
run(f'"{sys.executable}" -m uv pip install vllm==0.19.0 --torch-backend=auto --system -q')
run(f'"{sys.executable}" -m pip install -e . --no-cache-dir --quiet', cwd=REPO_DIR)
run(f'"{sys.executable}" -m pip install onnxruntime aiohttp numpy tqdm prettytable -q')

print("\nVERIFY IMPORTS:")
run(f'''"{sys.executable}" -c "import vllm; print('vllm version:', vllm.__version__)"''')
run(f'''"{sys.executable}" -c "import vllm_omni; print('vllm_omni path:', vllm_omni.__file__)"''')



+ "/usr/bin/python3" -m pip install -U uv -q
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 115.4 MB/s eta 0:00:00
+ "/usr/bin/python3" -m uv pip install vllm==0.19.0 --torch-backend=auto --system -q
+ "/usr/bin/python3" -m pip install -e . --no-cache-dir --quiet
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 321.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 350.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 270.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 344.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 192.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 321.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 264.4 MB/s eta 0:00:00
 

CompletedProcess(args='"/usr/bin/python3" -c "import vllm_omni; print(\'vllm_omni path:\', vllm_omni.__file__)"', returncode=0, stdout='vllm_omni path: /content/vllm-omni/vllm_omni/__init__.py\n', stderr='2026-04-09 19:56:33.387060: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.\n2026-04-09 19:56:33.405607: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered\nWARNING: All log messages before absl::InitializeLog() is called are written to STDERR\nE0000 00:00:1775764593.429835    2262 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered\nE0000 00:00:1775764593.43

In [ ]:
# Hugging Face login
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("HF login OK")


HF login OK


In [ ]:
import sys
mods = [m for m in list(sys.modules) if m.startswith("vllm_omni")]
for m in mods:
  del sys.modules[m]

In [ ]:
# !rm -rf /content/ming_bench_results

In [ ]:
# Benchmark settings
import json
import os
import time

PORT = 8091
MODEL = "inclusionAI/Ming-omni-tts-0.5B"
RESULTS_DIR = "/content/ming_bench_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
GPU_MONITOR_PROC = None

# Start with eager only. Add the non-eager configs after the serving path works.
CONFIGS_TO_RUN = [
    "sequential_eager",
    "async_chunk_eager",
]
# CONFIGS_TO_RUN = ["async_chunk_noneager"]
PRIMARY_COMPARISON = ("sequential_eager", "async_chunk_eager")

# Uncomment this when you want the full matrix:
# CONFIGS_TO_RUN = [
#     "async_chunk_eager",
#     "sequential_eager",
#     "async_chunk_noneager",
#     "sequential_noneager",
# ]

CONFIGS = {
    "sequential_eager": f"{REPO_DIR}/benchmarks/ming-tts/vllm_omni/configs/ming_tts_sequential_eager.yaml",
    "sequential_noneager": f"{REPO_DIR}/benchmarks/ming-tts/vllm_omni/configs/ming_tts_sequential_noneager.yaml",
    "async_chunk_eager": f"{REPO_DIR}/benchmarks/ming-tts/vllm_omni/configs/ming_tts_async_chunk_eager.yaml",
    "async_chunk_noneager": f"{REPO_DIR}/benchmarks/ming-tts/vllm_omni/configs/ming_tts_async_chunk_noneager.yaml",
}

# Keep concurrency at 1 on Colab.
NUM_PROMPTS = 10
NUM_WARMUPS = 2
MAX_CONCURRENCY = 1

# Match the repo's existing streaming Ming serving test as closely as possible:
# English prompts + streaming WAV response.
BENCH_EXTRA_ARGS = ""
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
PROMPTS_FILE = f"{RESULTS_DIR}/english_prompts_{RUN_TAG}.json"
GENERATIONS_ROOT = f"{RESULTS_DIR}/generations_{RUN_TAG}"
SAVE_WARMUP_GENERATIONS = True

ENGLISH_PROMPTS = [
    "The train leaves at seven thirty, so we should get to the station a little early.",
    "Please remember to bring your ID card tomorrow morning and arrive on time for the appointment.",
    "After the meeting, let us review the next phase and lock in the execution plan.",
    "Learning a new language takes patience, repetition, and the habit of speaking out loud.",
    "The wind outside has gone quiet, and the whole city feels like it is slowly falling asleep.",
    "If you want, I can tell you everything that happened today from the very beginning.",
    "This proposal still needs another pass, especially around the rollout details and timing.",
    "The sky turned a soft orange at sunset, and the last light stayed on the buildings for a while.",
    "I will stay here with you until you drift into the gentlest sleep.",
    "The product name is ridiculous on purpose, and that is exactly why people remember it.",
]

with open(PROMPTS_FILE, "w", encoding="utf-8") as f:
    json.dump(ENGLISH_PROMPTS, f, indent=2, ensure_ascii=False)

print("English prompts file:", PROMPTS_FILE)
print("Generation root:", GENERATIONS_ROOT)

# Keep GPU monitor off by default on Colab. Enable it only after one clean run.
ENABLE_GPU_MONITOR = False






English prompts file: /content/ming_bench_results/english_prompts_20260409_200630.json
Generation root: /content/ming_bench_results/generations_20260409_200630


In [ ]:
# Helper functions
def wait_for_server(port=PORT, timeout_s=300):
    start = time.time()
    while time.time() - start < timeout_s:
        response = subprocess.run(
            f"curl -s http://127.0.0.1:{port}/health",
            shell=True,
            text=True,
            capture_output=True,
        )
        if response.returncode == 0:
            print("Server is ready")
            return
        time.sleep(2)
    raise RuntimeError("Server did not become ready within timeout")


def start_server(config_name, config_path):
    log_path = f"{RESULTS_DIR}/{config_name}.log"
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONUNBUFFERED"] = "1"

    log_file = open(log_path, "w")
    cmd = [
        sys.executable,
        "-m",
        "vllm_omni.entrypoints.cli.main",
        "serve",
        MODEL,
        "--omni",
        "--port",
        str(PORT),
        "--stage-configs-path",
        config_path,
        "--log-stats",
    ]

    print("+", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    wait_for_server()
    return proc, log_file, log_path


def stop_server(proc, log_file):
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
    log_file.close()


def extract_server_error(log_path):
    if not log_path or not os.path.exists(log_path):
        return None

    with open(log_path, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for line in reversed(lines[-80:]):
        text = line.strip()
        if not text:
            continue
        if "ERROR" in text or "Traceback" in text or "RuntimeError" in text or "ValueError" in text:
            return text
    return None


def start_gpu_monitor():
    global GPU_MONITOR_PROC

    if not ENABLE_GPU_MONITOR:
        print("Skipping GPU monitor (ENABLE_GPU_MONITOR=False)")
        return None

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    cmd = [
        "bash",
        "tests/dfx/stability/scripts/resource_monitor.sh",
        "start",
        "--backend",
        "gpu",
    ]
    print("+", " ".join(cmd))
    GPU_MONITOR_PROC = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    time.sleep(2)
    return GPU_MONITOR_PROC


def finalize_gpu_monitor():
    global GPU_MONITOR_PROC

    if not ENABLE_GPU_MONITOR:
        return None

    if GPU_MONITOR_PROC is not None and GPU_MONITOR_PROC.poll() is None:
        GPU_MONITOR_PROC.terminate()
        try:
            GPU_MONITOR_PROC.wait(timeout=10)
        except subprocess.TimeoutExpired:
            GPU_MONITOR_PROC.kill()
            GPU_MONITOR_PROC.wait(timeout=5)
    GPU_MONITOR_PROC = None

    result = run(
        "bash tests/dfx/stability/scripts/resource_monitor.sh finalize --backend gpu",
        cwd=REPO_DIR,
        check=False,
    )
    bundle_dir = None
    for line in (result.stdout or "").splitlines():
        if line.startswith("GPU_MONITOR_BUNDLE_DIR="):
            bundle_dir = line.split("=", 1)[1].strip()
    return bundle_dir


def run_benchmark(config_name, extra_args=""):
    extra_args = extra_args.strip()
    rendered_extra_args = extra_args if extra_args else ""
    save_warmups_flag = "--save-warmups" if SAVE_WARMUP_GENERATIONS else ""
    audio_dir = f"{GENERATIONS_ROOT}/{config_name}"
    cmd = f'''
"{sys.executable}" benchmarks/ming-tts/vllm_omni/bench_tts_serve.py \
  --host 127.0.0.1 \
  --port {PORT} \
  --num-prompts {NUM_PROMPTS} \
  --max-concurrency {MAX_CONCURRENCY} \
  --num-warmups {NUM_WARMUPS} \
  --config-name {config_name} \
  --result-dir "{RESULTS_DIR}" \
  --prompts-file "{PROMPTS_FILE}" \
  --save-audio-dir "{audio_dir}" \
  {save_warmups_flag} \
  {rendered_extra_args}
'''
    return run(cmd, cwd=REPO_DIR)


def parse_stats_log(config_name, log_path):
    out_json = f"{RESULTS_DIR}/{config_name}_metrics.json"
    cmd = f'''
"{sys.executable}" benchmarks/ming-tts/vllm_omni/parse_log_stats.py \
  "{log_path}" \
  --output-json "{out_json}"
'''
    run(cmd, cwd=REPO_DIR)
    return out_json


def latest_bench_json(config_name):
    from pathlib import Path

    matches = sorted(Path(RESULTS_DIR).glob(f"bench_{config_name}_*.json"))
    if not matches:
        raise FileNotFoundError(f"No benchmark JSON found for {config_name}")
    return str(matches[-1])


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def summarize_config_result(config_name, info):
    status = info.get("status", "UNKNOWN")
    if status != "OK":
        return {
            "config": config_name,
            "status": status,
            "detail": info.get("error"),
        }

    bench_data = load_json(info["bench_json"])
    if not bench_data:
        return {
            "config": config_name,
            "status": "BENCH_FAILED",
            "detail": "empty benchmark json",
        }
    bench_row = bench_data[0]

    metrics_data = load_json(info["metrics_json"])
    stage0 = metrics_data.get("per_stage", {}).get("0", {})
    stage1 = metrics_data.get("per_stage", {}).get("1", {})
    edge01 = metrics_data.get("per_edge", {}).get("0->1", {})

    return {
        "config": config_name,
        "status": "OK",
        "mean_ttfp_ms": bench_row.get("mean_ttfp_ms"),
        "mean_e2e_ms": bench_row.get("mean_e2e_ms"),
        "mean_rtf": bench_row.get("mean_rtf"),
        "stage0_ms": stage0.get("mean_stage_gen_time_ms"),
        "stage1_ms": stage1.get("mean_stage_gen_time_ms"),
        "transfer_ms": edge01.get("mean_total_transfer_time_ms"),
        "transfer_mbps": edge01.get("mean_throughput_mbps"),
        "gpu_bundle": info.get("gpu_bundle"),
    }


def percent_delta(base, candidate):
    if base in (None, 0) or candidate is None:
        return None
    return ((candidate - base) / base) * 100.0


def build_async_vs_sequential_comparison(summary_rows):
    by_config = {row["config"]: row for row in summary_rows if row.get("status") == "OK"}
    sequential_name, async_name = PRIMARY_COMPARISON
    sequential = by_config.get(sequential_name)
    async_chunk = by_config.get(async_name)
    if sequential is None or async_chunk is None:
        return {
            "status": "INCOMPLETE",
            "sequential_config": sequential_name,
            "async_config": async_name,
        }

    comparison = {
        "status": "OK",
        "sequential_config": sequential_name,
        "async_config": async_name,
        "sequential": {
            "mean_ttfp_ms": sequential.get("mean_ttfp_ms"),
            "mean_e2e_ms": sequential.get("mean_e2e_ms"),
            "mean_rtf": sequential.get("mean_rtf"),
        },
        "async_chunk": {
            "mean_ttfp_ms": async_chunk.get("mean_ttfp_ms"),
            "mean_e2e_ms": async_chunk.get("mean_e2e_ms"),
            "mean_rtf": async_chunk.get("mean_rtf"),
        },
        "delta_pct": {
            "mean_ttfp_ms": percent_delta(sequential.get("mean_ttfp_ms"), async_chunk.get("mean_ttfp_ms")),
            "mean_e2e_ms": percent_delta(sequential.get("mean_e2e_ms"), async_chunk.get("mean_e2e_ms")),
            "mean_rtf": percent_delta(sequential.get("mean_rtf"), async_chunk.get("mean_rtf")),
        },
    }
    return comparison


def print_async_vs_sequential_comparison(comparison):
    def fmt(value, digits):
        if value is None:
            return "n/a"
        return f"{value:.{digits}f}"

    print("\n=== Ming Async Chunk vs Sequential ===")
    if comparison.get("status") != "OK":
        print(json.dumps(comparison, indent=2, ensure_ascii=False))
        return

    sequential = comparison["sequential"]
    async_chunk = comparison["async_chunk"]
    delta = comparison["delta_pct"]
    print(
        f"{'Mode':<14}{'Mean TTFP':>14}{'Mean E2E':>14}{'Mean RTF':>12}"
    )
    print(
        f"{'Sequential':<14}{fmt(sequential['mean_ttfp_ms'], 2):>14}{fmt(sequential['mean_e2e_ms'], 2):>14}{fmt(sequential['mean_rtf'], 3):>12}"
    )
    print(
        f"{'Async Chunk':<14}{fmt(async_chunk['mean_ttfp_ms'], 2):>14}{fmt(async_chunk['mean_e2e_ms'], 2):>14}{fmt(async_chunk['mean_rtf'], 3):>12}"
    )
    print(
        f"{'Delta %':<14}{fmt(delta['mean_ttfp_ms'], 2):>14}{fmt(delta['mean_e2e_ms'], 2):>14}{fmt(delta['mean_rtf'], 2):>12}"
    )


def cleanup_leftovers():
    run("pkill -f 'vllm_omni.entrypoints.cli.main serve' || true", check=False)
    run("pkill -f 'resource_monitor.sh start' || true", check=False)
    run("pkill -f 'nvidia-smi --query-gpu' || true", check=False)
    run(
        "rm -f tests/dfx/stability/gpu_monitor_data/current_run_id || true",
        cwd=REPO_DIR,
        check=False,
    )


In [ ]:
# # Run a single config first
# cleanup_leftovers()

# config_name = CONFIGS_TO_RUN[0]
# config_path = CONFIGS[config_name]

# proc = None
# log_file = None
# try:
#     proc, log_file, log_path = start_server(config_name, config_path)
#     start_gpu_monitor()
#     run_benchmark(config_name, BENCH_EXTRA_ARGS)
# finally:
#     if proc is not None and log_file is not None:
#         stop_server(proc, log_file)

# gpu_bundle = finalize_gpu_monitor()
# metrics_json = parse_stats_log(config_name, log_path)
# bench_json = latest_bench_json(config_name)

# print("bench_json:", bench_json)
# print("metrics_json:", metrics_json)
# print("gpu_bundle:", gpu_bundle)
# print("generation_dir:", f"{GENERATIONS_ROOT}/{config_name}")



In [ ]:

# Run the full selected matrix
cleanup_leftovers()

all_results = {}

for config_name in CONFIGS_TO_RUN:
    config_path = CONFIGS[config_name]
    proc = None
    log_file = None
    log_path = None
    print(f"\n===== {config_name} =====")

    try:
        proc, log_file, log_path = start_server(config_name, config_path)
    except Exception as exc:
        gpu_bundle = finalize_gpu_monitor()
        all_results[config_name] = {
            "status": "BOOT_FAILED",
            "error": str(exc),
            "detail": extract_server_error(log_path),
            "log_path": log_path,
            "gpu_bundle": gpu_bundle,
        }
        print(f"[BOOT_FAILED] {config_name}: {exc}")
        if proc is not None and log_file is not None:
            stop_server(proc, log_file)
        continue

    try:
        start_gpu_monitor()
        run_benchmark(config_name, BENCH_EXTRA_ARGS)
        bench_json = latest_bench_json(config_name)
    except Exception as exc:
        gpu_bundle = finalize_gpu_monitor()
        all_results[config_name] = {
            "status": "BENCH_FAILED",
            "error": str(exc),
            "detail": extract_server_error(log_path),
            "log_path": log_path,
            "gpu_bundle": gpu_bundle,
        }
        print(f"[BENCH_FAILED] {config_name}: {exc}")
        continue
    finally:
        if proc is not None and log_file is not None:
            stop_server(proc, log_file)

    try:
        metrics_json = parse_stats_log(config_name, log_path)
    except Exception as exc:
        gpu_bundle = finalize_gpu_monitor()
        all_results[config_name] = {
            "status": "PARSE_FAILED",
            "error": str(exc),
            "detail": extract_server_error(log_path),
            "bench_json": bench_json,
            "log_path": log_path,
            "gpu_bundle": gpu_bundle,
        }
        print(f"[PARSE_FAILED] {config_name}: {exc}")
        continue

    gpu_bundle = finalize_gpu_monitor()
    all_results[config_name] = {
        "status": "OK",
        "bench_json": bench_json,
        "metrics_json": metrics_json,
        "log_path": log_path,
        "gpu_bundle": gpu_bundle,
        "generation_dir": f"{GENERATIONS_ROOT}/{config_name}",
    }

all_results



+ pkill -f 'vllm_omni.entrypoints.cli.main serve' || true
+ pkill -f 'resource_monitor.sh start' || true
+ pkill -f 'nvidia-smi --query-gpu' || true
+ rm -f tests/dfx/stability/gpu_monitor_data/current_run_id || true

===== sequential_eager =====
+ /usr/bin/python3 -m vllm_omni.entrypoints.cli.main serve inclusionAI/Ming-omni-tts-0.5B --omni --port 8091 --stage-configs-path /content/vllm-omni/benchmarks/ming-tts/vllm_omni/configs/ming_tts_sequential_eager.yaml --log-stats
Server is ready
Skipping GPU monitor (ENABLE_GPU_MONITOR=False)
+ 
"/usr/bin/python3" benchmarks/ming-tts/vllm_omni/bench_tts_serve.py   --host 127.0.0.1   --port 8091   --num-prompts 10   --max-concurrency 1   --num-warmups 2   --config-name sequential_eager   --result-dir "/content/ming_bench_results"   --prompts-file "/content/ming_bench_results/english_prompts_20260409_200630.json"   --save-audio-dir "/content/ming_bench_results/generations_20260409_200630/sequential_eager"   --save-warmups   

  Warming up with 2

{'sequential_eager': {'status': 'OK',
  'bench_json': '/content/ming_bench_results/bench_sequential_eager_20260409_201015.json',
  'metrics_json': '/content/ming_bench_results/sequential_eager_metrics.json',
  'log_path': '/content/ming_bench_results/sequential_eager.log',
  'gpu_bundle': None,
  'generation_dir': '/content/ming_bench_results/generations_20260409_200630/sequential_eager'},
 'async_chunk_eager': {'status': 'OK',
  'bench_json': '/content/ming_bench_results/bench_async_chunk_eager_20260409_201304.json',
  'metrics_json': '/content/ming_bench_results/async_chunk_eager_metrics.json',
  'log_path': '/content/ming_bench_results/async_chunk_eager.log',
  'gpu_bundle': None,
  'generation_dir': '/content/ming_bench_results/generations_20260409_200630/async_chunk_eager'}}

In [ ]:
# Compact PR summary
summary_rows = []

for config_name, info in all_results.items():
    summary_rows.append(summarize_config_result(config_name, info))

summary_rows




[{'config': 'sequential_eager',
  'status': 'OK',
  'mean_ttfp_ms': 3289.982391400008,
  'mean_e2e_ms': 3292.180213100073,
  'mean_rtf': 0.5503928951657248,
  'stage0_ms': 2812.6691666666666,
  'stage1_ms': 609.5093333333333,
  'transfer_ms': None,
  'transfer_mbps': None,
  'gpu_bundle': None},
 {'config': 'async_chunk_eager',
  'status': 'OK',
  'mean_ttfp_ms': 1890.6289961999842,
  'mean_e2e_ms': 3210.3313616999462,
  'mean_rtf': 0.5379347730550441,
  'stage0_ms': 2959.972416666667,
  'stage1_ms': 3182.4546666666665,
  'transfer_ms': None,
  'transfer_mbps': None,
  'gpu_bundle': None}]

In [ ]:
# Async chunk vs sequential comparison
async_vs_sequential = build_async_vs_sequential_comparison(summary_rows)
print_async_vs_sequential_comparison(async_vs_sequential)
async_vs_sequential



=== Ming Async Chunk vs Sequential ===
Mode               Mean TTFP      Mean E2E    Mean RTF
Sequential           3289.98       3292.18       0.550
Async Chunk          1890.63       3210.33       0.538
Delta %               -42.53         -2.49       -2.26


{'status': 'OK',
 'sequential_config': 'sequential_eager',
 'async_config': 'async_chunk_eager',
 'sequential': {'mean_ttfp_ms': 3289.982391400008,
  'mean_e2e_ms': 3292.180213100073,
  'mean_rtf': 0.5503928951657248},
 'async_chunk': {'mean_ttfp_ms': 1890.6289961999842,
  'mean_e2e_ms': 3210.3313616999462,
  'mean_rtf': 0.5379347730550441},
 'delta_pct': {'mean_ttfp_ms': -42.53376549546053,
  'mean_e2e_ms': -2.486159508353711,
  'mean_rtf': -2.2634961715720316}}

In [ ]:
# Optional: save a merged summary JSON for the PR
summary_path = f"{RESULTS_DIR}/ming_benchmark_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "repo_url": REPO_URL,
            "branch": BRANCH,
            "configs_run": CONFIGS_TO_RUN,
            "num_prompts": NUM_PROMPTS,
            "num_warmups": NUM_WARMUPS,
            "max_concurrency": MAX_CONCURRENCY,
            "bench_extra_args": BENCH_EXTRA_ARGS,
            "prompts_file": PROMPTS_FILE,
            "generations_root": GENERATIONS_ROOT,
            "results": all_results,
            "summary_rows": summary_rows,
            "async_vs_sequential": async_vs_sequential,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved summary to:", summary_path)


Saved summary to: /content/ming_bench_results/ming_benchmark_summary.json


In [ ]:
# Dedicated comparison artifact for the PR table
comparison_path = f"{RESULTS_DIR}/ming_async_vs_sequential.json"
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "repo_url": REPO_URL,
            "branch": BRANCH,
            "configs_compared": list(PRIMARY_COMPARISON),
            "num_prompts": NUM_PROMPTS,
            "num_warmups": NUM_WARMUPS,
            "max_concurrency": MAX_CONCURRENCY,
            "bench_extra_args": BENCH_EXTRA_ARGS,
            "prompts_file": PROMPTS_FILE,
            "generations_root": GENERATIONS_ROOT,
            "comparison": async_vs_sequential,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved comparison to:", comparison_path)

Saved comparison to: /content/ming_bench_results/ming_async_vs_sequential.json


In [ ]:
import wave

with wave.open("/content/ming_bench_results/generations_20260409_200630/sequential_eager/concurrency_1/requests/request_0009.wav", "rb") as f:
    print("channels:", f.getnchannels())
    print("sample_rate:", f.getframerate())
    print("frames:", f.getnframes())


channels: 1
sample_rate: 44100
frames: 268128


In [ ]:
import wave

with wave.open("/content/ming_bench_results/generations_20260409_200630/async_chunk_eager/concurrency_1/requests/request_0009.wav", "rb") as f:
    print("channels:", f.getnchannels())
    print("sample_rate:", f.getframerate())
    print("frames:", f.getnframes())


channels: 1
sample_rate: 44100
frames: 268128


In [ ]:
soundfile="/content/ming_bench_results/generations_20260409_200630/sequential_eager/concurrency_1/requests/request_0009.wav"

In [ ]:
async_soundfile="/content/ming_bench_results/generations_20260409_200630/async_chunk_eager/concurrency_1/requests/request_0009.wav"

In [ ]:
from IPython.display import Audio, display

display(Audio(soundfile))
